# Chapter 12 — Submanifolds of an Affine Space

Companion notebook to Chapter 12 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Two computations come back all the time: the tangent space as the **kernel**
of the Jacobian matrix of the equations, and as the **image** (column space)
of the Jacobian matrix of a parametrization.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, jacfwd, jvp

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

def kernel(J, tol=1e-9):
    # orthonormal basis (columns) of the kernel of the matrix J
    J = J.reshape(-1, J.shape[-1]) if J.dim() > 1 else J.reshape(1, -1)
    _, s, Vh = torch.linalg.svd(J, full_matrices=True)
    r = int((s > tol * max(1.0, s.max().item())).sum())
    return Vh[r:].T

def colspace(A, tol=1e-9):
    # orthonormal basis (columns) of the image of the matrix A
    U, s, _ = torch.linalg.svd(A, full_matrices=False)
    return U[:, s > tol * max(1.0, s.max().item())]

def same_subspace(A, B, tol=1e-8):
    # A and B have orthonormal columns
    return A.shape == B.shape and (A @ A.T - B @ B.T).abs().max().item() < tol

def rank(A, tol=1e-9):
    s = torch.linalg.svdvals(A)
    return int((s > tol * max(1.0, s.max().item())).sum())

2.15.0+cu132


## 1. One tangent line, four computations

Example 12.4 and Theorem 12.6: the unit circle at `p = (0.6, 0.8)`,
described by a straightening (polar coordinates), an equation, a graph and a
parametrization. Each description gives the tangent space by its own
formula; the four answers are the same line.

In [2]:
p = torch.tensor([0.6, 0.8])
g = lambda x: (x @ x - 1).reshape(1)                                   # (b) equation
T_b = kernel(jacrev(g)(p))
psi = lambda th: torch.stack([torch.cos(th), torch.sin(th)])           # (d) parametrization
th0 = torch.acos(p[0])
T_d = colspace(jacfwd(psi)(th0).reshape(2, 1))
h = lambda u: torch.sqrt(1 - (0.6 + u) ** 2) - 0.8                     # (c) graph over the x-axis
dh = jacfwd(h)(torch.tensor(0.0))
T_c = colspace(torch.stack([torch.tensor(1.0), dh]).reshape(2, 1))
Phi = lambda x: torch.stack([torch.atan2(x[1], x[0]), x.norm() - 1])   # (a) straightening, E = first axis
T_a = colspace(torch.linalg.solve(jacrev(Phi)(p), torch.tensor([1.0, 0.0])).reshape(2, 1))
print("[dg(p)] =", jacrev(g)(p).tolist(), "  h'(0) =", round(dh.item(), 6))
print("tangent direction:", [round(v, 4) for v in T_b[:, 0].tolist()])
assert abs(dh.item() + 0.75) < 1e-12
assert same_subspace(T_a, T_b) and same_subspace(T_c, T_b) and same_subspace(T_d, T_b)
# the straightening sends the circle into the line E = R x {0}
th = torch.linspace(0.1, 3.0, 50)
assert max(abs(Phi(psi(t))[1].item()) for t in th) < 1e-12

[dg(p)] = [[1.2, 1.6]]   h'(0) = -0.75
tangent direction: [-0.8, 0.6]


## 2. An injective immersion that is not a parametrization

The pitfall of Section 12.1 (Figure 12.2) and Example 12.10: the figure eight
`t -> (sin 2t, sin t)` on `(0, 2 pi)`, and the corner.

In [3]:
eight = lambda t: torch.stack([torch.sin(2 * t), torch.sin(t)])
ts = torch.linspace(1e-3, 2 * math.pi - 1e-3, 20001)
speed = torch.sqrt(4 * torch.cos(2 * ts) ** 2 + torch.cos(ts) ** 2)
print("smallest speed on the interval:", round(speed.min().item(), 4), "(an immersion)")
assert speed.min() > 0.6
# injective: away from the two ends, parameters that differ by more than 0.05 give points that stay apart
keep = (ts > 0.2) & (ts < 2 * math.pi - 0.2)
tsub = ts[keep][::25]
sub = torch.stack([torch.sin(2 * tsub), torch.sin(tsub)], 1)
far = (tsub[:, None] - tsub[None, :]).abs() > 0.05
gap = torch.cdist(sub, sub)[far].min().item()
print("smallest distance between points whose parameters differ by more than 0.05:", round(gap, 4))
assert gap > 0.02
# the inverse is not continuous at the origin = eight(pi)
for t in (1e-3, 2 * math.pi - 1e-3):
    q = eight(torch.tensor(t))
    print(f"t = {t:.4f}: distance to eight(pi) = {q.norm().item():.4f}, distance of the parameters = {abs(t - math.pi):.3f}")
    assert q.norm() < 3e-3 and abs(t - math.pi) > 3
# two independent velocities at the origin, from curves that stay in the figure eight
on_eight = lambda x, y: x ** 2 - 4 * y ** 2 * (1 - y ** 2)
vel = []
for sign in (1.0, -1.0):
    c = lambda s: torch.stack([sign * 2 * s * torch.sqrt(1 - s ** 2), s])
    ss = torch.linspace(-0.3, 0.3, 61)
    assert max(abs(on_eight(*c(s)).item()) for s in ss) < 1e-12
    vel.append(jacfwd(c)(torch.tensor(0.0)))
print("velocities at the origin:", [v.tolist() for v in vel], " determinant:", torch.det(torch.stack(vel)).item())
assert abs(torch.det(torch.stack(vel)).item()) == 4.0
assert torch.allclose(jacfwd(eight)(torch.tensor(math.pi)), torch.tensor([2.0, -1.0]))

# the corner: t -> (t^3, |t|^3) is C^1 and a homeomorphism onto {y = |x|}, with velocity 0 at t = 0
corner = lambda t: torch.stack([t ** 3, t.abs() ** 3])
print("velocity of the corner parametrization at 0:", jacfwd(corner)(torch.tensor(0.0)).tolist())
assert jacfwd(corner)(torch.tensor(0.0)).norm() == 0

smallest speed on the interval: 0.696 (an immersion)
smallest distance between points whose parameters differ by more than 0.05: 0.0382
t = 0.0010: distance to eight(pi) = 0.0022, distance of the parameters = 3.141
t = 6.2822: distance to eight(pi) = 0.0022, distance of the parameters = 3.141
velocities at the origin: [[2.0, 1.0], [-2.0, 1.0]]  determinant: 4.0
velocity of the corner parametrization at 0: [0.0, 0.0]


## 3. First-order contact, conormal and normal spaces

A submanifold of dimension 2 in `R^4`, given by two equations.
Proposition 12.7: the distance from a point `x` of `M` to the tangent flat
is `o(|x - p|)`. Proposition 12.9: the projection onto the tangent space
for an inner product with Gram matrix `G`; the conormal space does not
depend on `G`, the normal space does.

In [4]:
g = lambda x: torch.stack([x @ x - 3.0, x[0] * x[1] - x[2] + torch.sin(x[3])])
p = torch.tensor([1.0, 1.0, 1.0, 0.0])
J = jacrev(g)(p)
print("g(p) =", g(p).tolist(), "  J =", J.tolist(), "  rank", rank(J))
assert g(p).abs().max() == 0 and rank(J) == 2
T = kernel(J)                                   # 4 x 2, orthonormal for the dot product
assert T.shape == (4, 2)

def back_to_M(x, steps=30):
    for _ in range(steps):
        Jx = jacrev(g)(x)
        x = x - Jx.T @ torch.linalg.solve(Jx @ Jx.T, g(x))
    return x

u = T @ torch.tensor([0.6, 0.8])                # a unit tangent vector
ratios = []
for eps in (0.2, 0.1, 0.05, 0.025):
    x = back_to_M(p + eps * u)
    assert g(x).abs().max() < 1e-12
    d = x - p
    dist = (d - T @ (T.T @ d)).norm()           # distance to the tangent flat (dot product)
    ratios.append((dist / d.norm()).item())
    print(f"|x - p| = {d.norm().item():.4f}   dist(x, p + T_pM) / |x - p| = {ratios[-1]:.5f}")
assert all(0.4 < ratios[k + 1] / ratios[k] < 0.6 for k in range(3))        # the ratio tends to 0, linearly here

A_ = torch.randn(4, 4); G = A_ @ A_.T + 4 * torch.eye(4)                    # another inner product on V
Gi = torch.linalg.inv(G)
Pi = torch.eye(4) - Gi @ J.T @ torch.linalg.solve(J @ Gi @ J.T, J)
assert torch.allclose(Pi @ Pi, Pi) and (J @ Pi).abs().max() < 1e-12        # a projection into ker J
assert torch.allclose(Pi @ T, T)                                            # the identity on T_pM
assert torch.allclose(G @ Pi, (G @ Pi).T)                                   # self-adjoint for G
N_dot, N_G = colspace(J.T), colspace(Gi @ J.T)
print("normal space for the dot product = normal space for G ?", same_subspace(N_dot, N_G))
assert not same_subspace(N_dot, N_G)
assert (T.T @ G @ N_G).abs().max() < 1e-12                                  # G-orthogonal to T_pM
P0 = torch.eye(4) - J.T @ torch.linalg.solve(J @ J.T, J)
assert torch.allclose(P0, T @ T.T)                                          # orthonormal basis: I - J^T (J J^T)^{-1} J

g(p) = [0.0, 0.0]   J = [[2.0, 2.0, 2.0, 0.0], [1.0, 1.0, -1.0, 1.0]]   rank 2
|x - p| = 0.1989   dist(x, p + T_pM) / |x - p| = 0.05961
|x - p| = 0.0999   dist(x, p + T_pM) / |x - p| = 0.02988
|x - p| = 0.0500   dist(x, p + T_pM) / |x - p| = 0.01494
|x - p| = 0.0250   dist(x, p + T_pM) / |x - p| = 0.00747
normal space for the dot product = normal space for G ? False


## 4. Stiefel manifolds and the orthogonal group

Theorem 12.11 and the pitfall that follows it, with Exercise 12.5: for
`St(5, 2)` the equations `X^T X = I` are 4 scalar equations of rank 3; the
tangent space has dimension `10 - 3 = 7`, and is made of the matrices
`X Omega + X_perp B`.

In [5]:
n, k = 5, 2
Qfull, _ = torch.linalg.qr(torch.randn(n, n))
X, Xp = Qfull[:, :k], Qfull[:, k:]
gst = lambda x: (x.reshape(n, k).T @ x.reshape(n, k) - torch.eye(k)).reshape(-1)
Jst = jacrev(gst)(X.reshape(-1))                                    # 4 x 10: the map into R^{k x k}
print("Jacobian of X^T X - I as a map into R^(k x k):", tuple(Jst.shape), " rank", rank(Jst),
      "  k(k+1)/2 =", k * (k + 1) // 2)
assert rank(Jst) == k * (k + 1) // 2 < k * k
Tst = kernel(Jst)
print("dimension of the tangent space:", Tst.shape[1], "  nk - k(k+1)/2 =", n * k - k * (k + 1) // 2)
assert Tst.shape[1] == n * k - k * (k + 1) // 2 == 7
# Dg(X) Z = X^T Z + Z^T X, and Z = X S / 2 is sent to S
Z = torch.randn(n, k); S = torch.randn(k, k); S = S + S.T
assert torch.allclose((Jst @ Z.reshape(-1)).reshape(k, k), X.T @ Z + Z.T @ X)
assert torch.allclose((Jst @ (X @ S / 2).reshape(-1)).reshape(k, k), S)
# Exercise 12.5
basis = []
for _ in range(12):
    Om = torch.randn(k, k); Om = Om - Om.T
    Zt = X @ Om + Xp @ torch.randn(n - k, k)
    assert (X.T @ Zt + Zt.T @ X).abs().max() < 1e-12
    basis.append(Zt.reshape(-1))
span = colspace(torch.stack(basis, 1))
print("dimension spanned by the matrices X Omega + X_perp B:", span.shape[1], "= 1 + 6")
assert same_subspace(span, Tst)

# O(3): dimension 3, tangent vectors Q Omega, and a curve that stays in the group
Q, _ = torch.linalg.qr(torch.randn(3, 3))
go = lambda x: (x.reshape(3, 3).T @ x.reshape(3, 3) - torch.eye(3)).reshape(-1)
assert kernel(jacrev(go)(Q.reshape(-1))).shape[1] == 3
Om = torch.randn(3, 3); Om = Om - Om.T
curve = lambda t: Q @ torch.linalg.matrix_exp(t * Om)
assert (curve(torch.tensor(0.7)).T @ curve(torch.tensor(0.7)) - torch.eye(3)).abs().max() < 1e-12
assert torch.allclose(jacfwd(curve)(torch.tensor(0.0)), Q @ Om)
print("O(3): tangent space of dimension 3; the curve Q exp(t Omega) has velocity Q Omega")

Jacobian of X^T X - I as a map into R^(k x k): (4, 10)  rank 3   k(k+1)/2 = 3
dimension of the tangent space: 7   nk - k(k+1)/2 = 7
dimension spanned by the matrices X Omega + X_perp B: 7 = 1 + 6
O(3): tangent space of dimension 3; the curve Q exp(t Omega) has velocity Q Omega


## 5. Matrices of fixed rank

Theorem 12.13 for `m = 5`, `n = 4`, `r = 2`: the Schur complement
criterion, the dimension `r(m + n - r) = 14`, the tangent space
`{H : H(ker A) in im A}`. Then Exercise 12.6 and Example 12.14.

In [6]:
m, n, r = 5, 4, 2
B, C = torch.randn(m, r), torch.randn(r, n)
A = B @ C
schur = lambda M: M[r:, r:] - M[r:, :r] @ torch.linalg.solve(M[:r, :r], M[:r, r:])
print("rank of A:", rank(A), "  norm of the Schur complement:", f"{schur(A).norm().item():.1e}")
A_other = A + 0.1 * torch.randn(m, n)
print("after a generic perturbation: rank", rank(A_other), "= r + rank of the Schur complement =",
      r, "+", rank(schur(A_other)))
assert schur(A).norm() < 1e-12 and rank(A_other) == r + rank(schur(A_other))

mu = lambda z: (z[:m * r].reshape(m, r) @ z[m * r:].reshape(r, n)).reshape(-1)
Jmu = jacrev(mu)(torch.cat([B.reshape(-1), C.reshape(-1)]))         # 20 x 18
print("differential of (B, A) -> BA: rank", rank(Jmu), "= r(m+n-r) =", r * (m + n - r),
      "  kernel of dimension", Jmu.shape[1] - rank(Jmu), "= r^2")
assert rank(Jmu) == r * (m + n - r) == 14 and Jmu.shape[1] - rank(Jmu) == r * r
# the kernel is made of the pairs (B G, -G A)
Gm = torch.randn(r, r)
assert (Jmu @ torch.cat([(B @ Gm).reshape(-1), (-Gm @ C).reshape(-1)])).abs().max() < 1e-12

U, s, Vh = torch.linalg.svd(A)
PU = U[:, :r] @ U[:, :r].T                                          # projection onto im A
PK = torch.eye(n) - Vh[:r].T @ Vh[:r]                               # projection onto ker A
cond = lambda H: ((torch.eye(m) - PU) @ H @ PK).norm().item()       # 0 iff H(ker A) is inside im A
image = colspace(Jmu)
assert max(cond(image[:, i].reshape(m, n)) for i in range(image.shape[1])) < 1e-12
# the condition H(ker A) in im A defines a space of dimension mn - (m-r)(n-r) = 14: the same space
E = torch.eye(m * n)
Lcond = torch.stack([((torch.eye(m) - PU) @ E[i].reshape(m, n) @ PK).reshape(-1) for i in range(m * n)], 1)
assert same_subspace(kernel(Lcond), image)

sig = lambda M: torch.linalg.svdvals(M)[r].item()                   # the (r+1)-th singular value
H_tan = torch.randn(m, r) @ C + B @ torch.randn(r, n)
H_gen = torch.randn(m, n)
for name, H in (("tangent H ", H_tan), ("generic H ", H_gen)):
    a, b = sig(A + 1e-2 * H), sig(A + 1e-3 * H)
    print(f"{name}: sigma_(r+1)(A + tH) = {a:.2e} at t = 1e-2, {b:.2e} at t = 1e-3  (ratio {a / b:.0f})")
assert 80 < sig(A + 1e-2 * H_tan) / sig(A + 1e-3 * H_tan) < 125     # second order
assert 8 < sig(A + 1e-2 * H_gen) / sig(A + 1e-3 * H_gen) < 12       # first order

# Exercise 12.6
A2 = torch.tensor([[1.0, 2.0], [2.0, 4.0]])
ddet = jacrev(torch.det)(A2)
print("d(det)(A) =", [[round(v, 6) for v in row] for row in ddet.tolist()])
assert torch.allclose(ddet, torch.tensor([[4.0, -2.0], [-2.0, 1.0]]))
eq = lambda H: 4 * H[0, 0] - 2 * H[0, 1] - 2 * H[1, 0] + H[1, 1]
H1 = torch.tensor([[0.0, 1.0], [1.0, 4.0]])
assert eq(H1) == 0 and eq(torch.eye(2)) == 5
assert abs((ddet * H1).sum().item()) < 1e-12 and abs((ddet * torch.eye(2)).sum().item() - 5) < 1e-12
t = 1e-3
assert abs(torch.det(A2 + t * torch.eye(2)).item() - (5 * t + t * t)) < 1e-12
print("det(A + t H1) =", f"{torch.det(A2 + t * H1).item():.2e}", "(order t^2);  det(A + t I) =",
      f"{torch.det(A2 + t * torch.eye(2)).item():.2e}", "(order t)")

# Example 12.14: symmetric matrices of rank one
x, y = torch.tensor([1.0, 2.0]), torch.tensor([0.5, -1.0])
detsym = lambda v: v[0] * v[2] - v[1] ** 2                          # coordinates (a, b, c)
assert torch.allclose(jacrev(detsym)(torch.tensor([1.0, 2.0, 4.0])), torch.tensor([4.0, -4.0, 1.0]))
Hs = torch.outer(x, y) + torch.outer(y, x)
for t in (1e-2, 1e-3):
    assert abs(torch.det(torch.outer(x, x) + t * Hs).item() + t * t * (x[0] * y[1] - x[1] * y[0]).item() ** 2) < 1e-12
print("det(x x^T + t (x y^T + y x^T)) = -t^2 (x1 y2 - x2 y1)^2: tangent to the cone")

rank of A: 2   norm of the Schur complement: 1.5e-15
after a generic perturbation: rank 4 = r + rank of the Schur complement = 2 + 2
differential of (B, A) -> BA: rank 14 = r(m+n-r) = 14   kernel of dimension 4 = r^2
tangent H : sigma_(r+1)(A + tH) = 2.49e-04 at t = 1e-2, 2.51e-06 at t = 1e-3  (ratio 99)
generic H : sigma_(r+1)(A + tH) = 2.63e-02 at t = 1e-2, 2.64e-03 at t = 1e-3  (ratio 10)
d(det)(A) = [[4.0, -2.0], [-2.0, 1.0]]
det(A + t H1) = -1.00e-06 (order t^2);  det(A + t I) = 5.00e-03 (order t)
det(x x^T + t (x y^T + y x^T)) = -t^2 (x1 y2 - x2 y1)^2: tangent to the cone


## 6. Maps on a submanifold

Proposition 12.18 (the differential is the restriction of the differential
of any extension), the pitfall of Section 12.4 (two extensions, two ambient
gradients, one projected gradient), Theorem 12.19 (critical points),
Corollary 12.17 (changes of parameters) and Example 12.21 (normalization).

In [7]:
F1 = lambda x: x[2]
F2 = lambda x: x[2] * (x @ x)
x = torch.tensor([2.0, -1.0, 2.0]) / 3                              # a point of the unit sphere
g1, g2 = jacrev(F1)(x), jacrev(F2)(x)
Pi = torch.eye(3) - torch.outer(x, x)
print("gradient of F1:", g1.tolist(), "\ngradient of F2:", [round(v, 4) for v in g2.tolist()])
print("difference / x:", [round(v, 4) for v in ((g2 - g1) / x).tolist()], " (a normal vector)")
assert torch.allclose(g2 - g1, 2 * x[2] * x)
assert torch.allclose(Pi @ g1, Pi @ g2) and torch.allclose(Pi @ g1, torch.tensor([0.0, 0.0, 1.0]) - x[2] * x)
# the differential along a curve of the sphere does not depend on the extension
w = Pi @ torch.tensor([0.3, 1.0, -0.5])
c = lambda t: (x + t * w) / (x + t * w).norm()
assert torch.allclose(jacfwd(c)(torch.tensor(0.0)), w)
for F, gr in ((F1, g1), (F2, g2)):
    assert abs(jacfwd(lambda t: F(c(t)))(torch.tensor(0.0)).item() - (gr @ w).item()) < 1e-12
# critical points of x3 on the sphere: the poles, with the multiplier -1/2 at the north pole
north = torch.tensor([0.0, 0.0, 1.0])
assert ((torch.eye(3) - torch.outer(north, north)) @ jacrev(F1)(north)).norm() == 0
assert torch.allclose(jacrev(F1)(north) + (-0.5) * jacrev(lambda v: v @ v - 1)(north), torch.zeros(3))

# two parametrizations of the sphere (stereographic projections) and their change of parameters
psiN = lambda t: torch.cat([2 * t, (t @ t - 1).reshape(1)]) / (t @ t + 1)
psiS = lambda t: torch.cat([2 * t, (1 - t @ t).reshape(1)]) / (t @ t + 1)
invS = lambda q: q[:2] / (1 + q[2])
t0 = torch.tensor([0.7, -1.2])
q = psiN(t0); JN = jacfwd(psiN)(t0)
assert abs((q @ q).item() - 1) < 1e-12 and rank(JN) == 2 and (q @ JN).abs().max() < 1e-12
change = lambda t: invS(psiN(t))
assert torch.allclose(change(t0), t0 / (t0 @ t0))
print("change of parameters t -> t/|t|^2, Jacobian determinant", round(torch.det(jacfwd(change)(t0)).item(), 4))
assert abs(torch.det(jacfwd(change)(t0)).item()) > 0.1
assert same_subspace(colspace(JN), colspace(jacfwd(psiS)(change(t0))))   # the same tangent plane, two bases

# normalization, for a general inner product
A_ = torch.randn(4, 4); G = A_ @ A_.T + torch.eye(4)
nrm = lambda v: torch.sqrt(v @ G @ v)
pi = lambda v: v / nrm(v)
v = torch.randn(4); wv = pi(v)
Dpi = jacrev(pi)(v)
assert torch.allclose(Dpi, (torch.eye(4) - torch.outer(wv, G @ wv)) / nrm(v))
assert rank(Dpi) == 3 and (Dpi @ v).abs().max() < 1e-12 and (wv @ G @ Dpi).abs().max() < 1e-12
print("D pi(v): rank 3, kernel R v, image orthogonal to w = pi(v)")

gradient of F1: [0.0, 0.0, 1.0] 
gradient of F2: [0.8889, -0.4444, 1.8889]
difference / x: [1.3333, 1.3333, 1.3333]  (a normal vector)
change of parameters t -> t/|t|^2, Jacobian determinant -0.2685


D pi(v): rank 3, kernel R v, image orthogonal to w = pi(v)


## 7. Machine learning: normalized weights, orthogonal weights, low rank

The three boxes of Section 12.5 and Exercise 12.8.

In [8]:
# weight normalization
torch.manual_seed(3)
Xd, yd = torch.randn(30, 6), torch.randn(30)
ell = lambda w: ((Xd @ w - yd) ** 2).mean()
gamma = 1.7
L = lambda v: ell(gamma * v / v.norm())
v = torch.randn(6); w = v / v.norm()
gv = jacrev(L)(v)
formula = gamma / v.norm() * (torch.eye(6) - torch.outer(w, w)) @ jacrev(ell)(gamma * w)
assert torch.allclose(gv, formula) and abs((gv @ v).item()) < 1e-12
eta = 0.3
v_new = v - eta * gv
print("gradient orthogonal to v;  |v|^2 before and after a step:", round((v @ v).item(), 6), round((v_new @ v_new).item(), 6))
assert abs((v_new @ v_new).item() - (v @ v).item() - eta ** 2 * (gv @ gv).item()) < 1e-12

# Cayley transform
Om = torch.randn(4, 4); Om = Om - Om.T
cayley = lambda O: (torch.eye(4) - O) @ torch.linalg.inv(torch.eye(4) + O)
Q = cayley(Om)
assert torch.allclose(Q.T @ Q, torch.eye(4)) and torch.allclose(cayley(Q), Om)
_, dQ = jvp(cayley, (torch.zeros(4, 4),), (Om,))
assert torch.allclose(dQ, -2 * Om)
print("Cayley transform: orthogonal, its own inverse formula, differential at 0 equal to -2 Omega; det Q =",
      round(torch.det(Q).item(), 6))

# a gradient step leaves the constraint set
Wst, _ = torch.linalg.qr(torch.randn(5, 2))
step = Wst - 0.1 * torch.randn(5, 2)
print("after a step, |W^T W - I| =", round((step.T @ step - torch.eye(2)).norm().item(), 4))

# low-rank adapters
m, n, r = 1000, 1000, 8
print("parameters of the adapter:", r * (m + n), "  dimension of the rank-r matrices:", r * (m + n - r),
      "  entries of the full matrix:", m * n)
assert (r * (m + n), r * (m + n - r), m * n) == (16000, 15936, 1000000)
m, n, r = 5, 4, 2
W0, Tg = torch.randn(m, n), torch.randn(m, n)
loss = lambda B, A: 0.5 * ((W0 + B @ A - Tg) ** 2).sum()
A0 = torch.randn(r, n); B0 = torch.zeros(m, r)
mu = lambda z: (z[:m * r].reshape(m, r) @ z[m * r:].reshape(r, n)).reshape(-1)
J0 = jacrev(mu)(torch.cat([B0.reshape(-1), A0.reshape(-1)]))
gB, gA = jacrev(loss, argnums=(0, 1))(B0, A0)
print("at B = 0: rank of the differential of (B, A) -> BA is", rank(J0), "= m r;  |dL/dA| =", gA.norm().item(),
      "  |dL/dB| =", round(gB.norm().item(), 3))
assert rank(J0) == m * r and gA.norm() == 0 and gB.norm() > 0
# the same model, two pairs (B, A), two different gradient steps
B1, A1 = torch.randn(m, r), torch.randn(r, n)
def one_step(B, A, eta=0.01):
    gB, gA = jacrev(loss, argnums=(0, 1))(B, A)
    return (B - eta * gB) @ (A - eta * gA)
s = 3.0
assert torch.allclose(B1 @ A1, (s * B1) @ (A1 / s))
diff = (one_step(B1, A1) - one_step(s * B1, A1 / s)).norm().item()
print("same product before the step; distance between the products after one step:", round(diff, 4))
assert diff > 1e-3

gradient orthogonal to v;  |v|^2 before and after a step: 6.426046 6.495001
Cayley transform: orthogonal, its own inverse formula, differential at 0 equal to -2 Omega; det Q = 1.0
after a step, |W^T W - I| = 0.1924
parameters of the adapter: 16000   dimension of the rank-r matrices: 15936   entries of the full matrix: 1000000
at B = 0: rank of the differential of (B, A) -> BA is 10 = m r;  |dL/dA| = 0.0   |dL/dB| = 8.307
same product before the step; distance between the products after one step: 2.4878


## 8. Machine learning: the manifold hypothesis

A "data set" on a submanifold of dimension 2 of `R^8`: the torus of
Exercise 12.7, placed in `R^8` by a linear isometry. The decoder is the
parametrization; its Jacobian gives the tangent space. From samples near a
point, the singular value decomposition of the displacements recovers the
dimension and, as the radius shrinks, the tangent space.

In [9]:
torch.manual_seed(4)
R8, _ = torch.linalg.qr(torch.randn(8, 3))
torus = lambda z: torch.stack([(2 + torch.cos(z[1])) * torch.cos(z[0]),
                               (2 + torch.cos(z[1])) * torch.sin(z[0]), torch.sin(z[1])])
decoder = lambda z: R8 @ torus(z)
z0 = torch.tensor([0.7, 1.1]); p = decoder(z0)
Jd = jacfwd(decoder)(z0)                                   # 8 x 2: two forward-mode columns
T_true = colspace(Jd)
print("Jacobian of the decoder:", tuple(Jd.shape), " rank", rank(Jd))
angles = []
for rho in (0.4, 0.2, 0.1, 0.05):
    Zs = z0 + rho * torch.rand(400, 2)                     # neighbours, on one side of z0
    D = torch.stack([decoder(z) for z in Zs]) - p          # displacements x_j - p
    _, s, Vh = torch.linalg.svd(D / math.sqrt(len(D)), full_matrices=False)
    T_est = Vh[:2].T
    sin_angle = torch.linalg.matrix_norm((torch.eye(8) - T_true @ T_true.T) @ T_est, 2).item()
    angles.append(sin_angle)
    print(f"rho = {rho:4.2f}: singular values {[round(v, 4) for v in s[:4].tolist()]}  "
          f"sine of the angle with T_pM = {sin_angle:.4f}")
    assert s[2] < 0.25 * s[1] and s[3] < 1e-10             # two large values: dimension 2 (the torus spans only 3 directions)
assert all(angles[k + 1] < 0.7 * angles[k] for k in range(3)) and angles[-1] < 0.03

Jacobian of the decoder: (8, 2)  rank 2
rho = 0.40: singular values [0.5644, 0.1366, 0.0113, 0.0]  sine of the angle with T_pM = 0.1641
rho = 0.20: singular values [0.289, 0.0694, 0.0031, 0.0]  sine of the angle with T_pM = 0.0768
rho = 0.10: singular values [0.1531, 0.0352, 0.0009, 0.0]  sine of the angle with T_pM = 0.0392
rho = 0.05: singular values [0.0734, 0.0172, 0.0002, 0.0]  sine of the angle with T_pM = 0.0190


## 9. The open simplex and the exercises

Example 12.12 and Exercise 12.9 (softmax), then Exercises 12.1, 12.3, 12.4
and 12.7.

In [10]:
# softmax: kernel spanned by (1, ..., 1), image = vectors of sum zero; log-odds invert the restriction to z_n = 0
z = torch.tensor([0.3, -1.0, 0.5, 2.0])
sm = lambda z: torch.softmax(z, 0)
Js = jacrev(sm)(z); pz = sm(z)
assert torch.allclose(Js, torch.diag(pz) - torch.outer(pz, pz))
assert rank(Js) == 3 and (Js @ torch.ones(4)).abs().max() < 1e-12 and (torch.ones(4) @ Js).abs().max() < 1e-12
s_ = lambda zp: sm(torch.cat([zp, torch.zeros(1)]))
zp = torch.tensor([0.4, -0.2, 1.3]); q = s_(zp)
assert torch.allclose(torch.log(q[:3] / q[3]), zp) and rank(jacrev(s_)(zp)) == 3
print("softmax: rank 3 on R^4; restricted to z_4 = 0 it is an immersion, inverted by the log-odds")

# Exercise 12.1
gh = lambda x: x[0] ** 2 + x[1] ** 2 - x[2] ** 2 - 1
ph = torch.ones(3)
assert torch.allclose(jacrev(gh)(ph), torch.tensor([2.0, 2.0, -2.0]))
for sv in torch.linspace(-3, 3, 13):
    for pt in (torch.stack([torch.tensor(1.0), sv, sv]), torch.stack([sv, torch.tensor(1.0), sv])):
        assert abs(gh(pt).item()) < 1e-12 and abs((pt[0] + pt[1] - pt[2] - 1).item()) < 1e-12
print("Exercise 12.1: the lines (1, s, s) and (s, 1, s) lie in the hyperboloid and in the flat x + y - z = 1")

# Exercise 12.3
cubic = lambda t: torch.stack([t, t ** 2, t ** 3])
gc = lambda x: torch.stack([x[1] - x[0] ** 2, x[2] - x[0] ** 3])
Jc = jacrev(gc)(torch.ones(3))
assert torch.allclose(Jc, torch.tensor([[-2.0, 1.0, 0.0], [-3.0, 0.0, 1.0]]))
assert same_subspace(kernel(Jc), colspace(jacfwd(cubic)(torch.tensor(1.0)).reshape(3, 1)))
assert torch.allclose(jacfwd(cubic)(torch.tensor(1.0)), torch.tensor([1.0, 2.0, 3.0]))
print("Exercise 12.3: kernel of the equations = image of the parametrization = span (1, 2, 3)")

# Exercise 12.4
M_ = torch.randn(3, 3); M_ = M_ / torch.det(M_).abs() ** (1 / 3)
if torch.det(M_) < 0: M_[0] = -M_[0]
H = torch.randn(3, 3)
assert abs(torch.det(M_).item() - 1) < 1e-12
assert abs((jacrev(torch.det)(M_) * H).sum().item() - torch.trace(torch.linalg.solve(M_, H)).item()) < 1e-10
print("Exercise 12.4: d(det)(A) H = tr(A^{-1} H) on SL(3)")

# Exercise 12.7
gt = lambda x: (torch.sqrt(x[0] ** 2 + x[1] ** 2) - 2) ** 2 + x[2] ** 2 - 1
zz = torch.tensor([0.0, 0.0])
assert torch.allclose(torus(zz), torch.tensor([3.0, 0.0, 0.0]))
assert torch.allclose(jacrev(gt)(torus(zz)), torch.tensor([2.0, 0.0, 0.0]))
assert torch.allclose(jacfwd(torus)(zz), torch.tensor([[0.0, 0.0], [3.0, 0.0], [0.0, 1.0]]))
for _ in range(100):
    zr = 6.28 * torch.rand(2)
    Jt = jacfwd(torus)(zr)
    assert abs(gt(torus(zr)).item()) < 1e-12 and abs((Jt[:, 0] @ Jt[:, 1]).item()) < 1e-12
    assert abs(Jt[:, 0].norm().item() - (2 + math.cos(zr[1].item()))) < 1e-12 and abs(Jt[:, 1].norm().item() - 1) < 1e-12
print("Exercise 12.7: the torus; tangent space at (3, 0, 0) is {v1 = 0}, by the kernel and by the image")

softmax: rank 3 on R^4; restricted to z_4 = 0 it is an immersion, inverted by the log-odds
Exercise 12.1: the lines (1, s, s) and (s, 1, s) lie in the hyperboloid and in the flat x + y - z = 1
Exercise 12.3: kernel of the equations = image of the parametrization = span (1, 2, 3)
Exercise 12.4: d(det)(A) H = tr(A^{-1} H) on SL(3)


Exercise 12.7: the torus; tangent space at (3, 0, 0) is {v1 = 0}, by the kernel and by the image


## Summary

* The tangent space is the kernel of the Jacobian matrix of the equations
  and the column space of the Jacobian matrix of a parametrization; the two
  computations agree.
* An injective immersion can fail to describe a submanifold: its inverse
  must be continuous.
* The conormal space is the row space of the Jacobian of the equations; the
  normal space and the projection depend on the inner product.
* `St(n, k)` has dimension `nk - k(k+1)/2`, the matrices of rank `r` have
  dimension `r(m+n-r)`; the factorization `BA` over-counts by `r^2` and is
  singular at `B = 0`.
* The ambient gradient of a function known only on `M` depends on the
  extension; its projection on the tangent space does not.